# U11 — 주동작 원판 초기 후보 자동 선택

Dataset V2 detector가 만든 첫 후보 frame에서 주동작 원판을 자동 선택하고 기존 수동 선택과 비교합니다.

## 1. 목표와 범위

- 후보가 하나면 자동 선택합니다.
- 후보가 여러 개면 주동작자 중심과의 거리가 명확히 가까운 후보만 선택합니다.
- 거리가 비슷하거나 Pose 중심이 없으면 억지로 고르지 않고 `confirmation_required`로 남깁니다.
- 새 모델을 학습하지 않고 기존 detector·Pose·tracker 결과를 재사용합니다.
- 이번 Task 1은 A01/B01/C01 세 pilot만 검증합니다.

In [ ]:
from pathlib import Path
import json
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snatch_analysis.pipeline import save_json_atomically
from snatch_analysis.plate import (
    save_plate_track,
    select_initial_plate_candidate,
    select_primary_plate_track,
)
from snatch_analysis.video import read_video_frame

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

## 2. 실험 설정

`U11_AMBIGUITY_MARGIN`은 가장 가까운 후보와 두 번째 후보의 상대 거리 차이가 어느 정도여야 자동 선택할지 정합니다. 기본 0.10은 거리 차이가 10% 이하면 사용자 확인을 요청한다는 뜻입니다. 다른 값은 이번 기준선 결과를 본 뒤 별도 실험에서만 바꿉니다.

In [ ]:
RUN_U11_AUTO_SELECTION = False
U11_AMBIGUITY_MARGIN = 0.10
U11_VIDEO_IDS = ('a01_sn_f_01', 'b01_sn_f_01', 'c01_sn_f_01')

EVALUATION_SPLIT_PATH = PROJECT_ROOT / 'data/splits/evaluation_videos.csv'
U9_POSE_ROOT = PROJECT_ROOT / 'artifacts/u9_baseline_conf010'
U10_EVALUATION_ROOT = PROJECT_ROOT / 'artifacts/u10_plate_dataset_v2/evaluation'
U11_ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection'
U11_SELECTION_PATH = U11_ARTIFACT_ROOT / 'initial_selections.csv'
U11_SUMMARY_PATH = U11_ARTIFACT_ROOT / 'run_summary.json'
U11_CONTACT_SHEET_PATH = U11_ARTIFACT_ROOT / 'initial_selection_contact_sheet.png'

if not 0 <= U11_AMBIGUITY_MARGIN <= 1:
    raise ValueError('U11_AMBIGUITY_MARGIN은 0 이상 1 이하여야 합니다.')

## 3. 저장된 후보·Pose·수동 기준 확인

U10에서 만든 raw 후보와 수동 초기 선택, U9에서 저장한 주동작자 bbox 중심을 읽습니다. 모델을 다시 추론하지 않습니다.

In [ ]:
evaluation_split = pd.read_csv(EVALUATION_SPLIT_PATH)
pilot_rows = (
    evaluation_split.loc[evaluation_split['video_id'].isin(U11_VIDEO_IDS)]
    .set_index('video_id')
    .loc[list(U11_VIDEO_IDS)]
    .reset_index()
)
if len(pilot_rows) != len(U11_VIDEO_IDS):
    raise ValueError('U11 pilot 영상 metadata가 모두 필요합니다.')
if set(pilot_rows['read_status']) != {'ok'}:
    raise ValueError('읽기 오류가 있는 pilot 영상은 사용할 수 없습니다.')

required_paths = []
for video_id in U11_VIDEO_IDS:
    required_paths.extend([
        U10_EVALUATION_ROOT / video_id / 'plate_candidates.csv',
        U10_EVALUATION_ROOT / video_id / 'run_summary.json',
        U9_POSE_ROOT / video_id / 'pose_keypoints.csv',
    ])
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('U11 입력 artifact가 없습니다:\n' + '\n'.join(missing_paths))
print(f'U11 입력 확인: {len(U11_VIDEO_IDS)}영상, 파일 {len(required_paths)}개')

## 4. 초기 후보 자동 선택과 tracker 연결

`RUN_U11_AUTO_SELECTION=True`로 바꾸면 자동 선택 결과를 기존 U10 수동 선택과 비교하고, 선택 성공 영상만 기존 tracker에 전달합니다.

In [ ]:
u11_selections = pd.DataFrame()
if not RUN_U11_AUTO_SELECTION:
    print('U11 실행 대기: RUN_U11_AUTO_SELECTION=True로 바꾸세요.')
else:
    U11_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    for stale_path in (U11_SELECTION_PATH, U11_SUMMARY_PATH, U11_CONTACT_SHEET_PATH):
        stale_path.unlink(missing_ok=True)
    selection_rows = []
    total_frame_count = 0
    track_evaluated_frame_count = 0
    total_ok_frames = 0
    metadata_by_video = pilot_rows.set_index('video_id')
    for video_id in U11_VIDEO_IDS:
        metadata = metadata_by_video.loc[video_id]
        candidates = pd.read_csv(
            U10_EVALUATION_ROOT / video_id / 'plate_candidates.csv'
        )
        pose_keypoints = pd.read_csv(
            U9_POSE_ROOT / video_id / 'pose_keypoints.csv',
            usecols=['frame_index', 'bbox_center_x', 'bbox_center_y'],
        )
        lifter_centers = (
            pose_keypoints[['frame_index', 'bbox_center_x', 'bbox_center_y']]
            .dropna()
            .drop_duplicates('frame_index')
            .rename(columns={
                'bbox_center_x': 'center_x_px',
                'bbox_center_y': 'center_y_px',
            })
        )
        selection = select_initial_plate_candidate(
            candidates,
            lifter_centers=lifter_centers,
            ambiguity_margin=U11_AMBIGUITY_MARGIN,
        )
        manual_summary = json.loads(
            (U10_EVALUATION_ROOT / video_id / 'run_summary.json').read_text(
                encoding='utf-8'
            )
        )['plate']
        manual_frame_value = manual_summary.get('initial_frame_index')
        manual_candidate_value = manual_summary.get('initial_candidate_id')
        manual_seed_available = (
            manual_frame_value is not None and not pd.isna(manual_frame_value)
            and manual_candidate_value is not None and not pd.isna(manual_candidate_value)
        )
        manual_frame_index = int(manual_frame_value) if manual_seed_available else None
        manual_candidate_id = int(manual_candidate_value) if manual_seed_available else None
        selection.update({
            'video_id': video_id,
            'group': str(metadata['group']),
            'manual_frame_index': manual_frame_index,
            'manual_candidate_id': manual_candidate_id,
            'matches_manual': (
                selection['status'] == 'selected'
                and selection['frame_index'] == manual_frame_index
                and selection['candidate_id'] == manual_candidate_id
            ) if manual_seed_available else None,
        })
        track_path = U11_ARTIFACT_ROOT / video_id / 'plate_track.csv'
        track_path.unlink(missing_ok=True)
        total_frame_count += int(metadata['frame_count'])
        if selection['status'] == 'selected':
            track = select_primary_plate_track(
                candidates,
                frame_count=int(metadata['frame_count']),
                initial_frame_index=int(selection['frame_index']),
                initial_candidate_id=int(selection['candidate_id']),
                lifter_centers=lifter_centers,
                fps=float(metadata['fps']),
            )
            save_plate_track(track, track_path)
            selection['track_ok_frame_count'] = int(track['status'].eq('ok').sum())
            track_evaluated_frame_count += int(metadata['frame_count'])
            total_ok_frames += int(selection['track_ok_frame_count'])
        else:
            selection['track_ok_frame_count'] = None
        selection_rows.append(selection)

    u11_selections = pd.DataFrame(selection_rows)
    u11_selections.to_csv(U11_SELECTION_PATH, index=False)
    selected_count = int(u11_selections['status'].eq('selected').sum())
    manual_comparison_mask = u11_selections['matches_manual'].notna()
    manual_comparison_count = int(manual_comparison_mask.sum())
    manual_match_count = int(u11_selections['matches_manual'].eq(True).sum())
    wrong_auto_selection_count = int((
        u11_selections['status'].eq('selected')
        & manual_comparison_mask
        & ~u11_selections['matches_manual'].eq(True)
    ).sum())
    confirmation_count = int(
        u11_selections['status'].eq('confirmation_required').sum()
    )
    not_available_count = int(u11_selections['status'].eq('not_available').sum())
    u11_summary = {
        'status': 'complete',
        'video_count': len(u11_selections),
        'selected_count': selected_count,
        'manual_comparison_count': manual_comparison_count,
        'manual_match_count': manual_match_count,
        'wrong_auto_selection_count': wrong_auto_selection_count,
        'confirmation_required_count': confirmation_count,
        'not_available_count': not_available_count,
        'manual_review_rate': (confirmation_count + not_available_count) / len(u11_selections),
        'track_ok_frame_count': total_ok_frames,
        'track_evaluated_frame_count': track_evaluated_frame_count,
        'track_not_evaluated_frame_count': total_frame_count - track_evaluated_frame_count,
        'total_frame_count': total_frame_count,
        'track_coverage': (
            total_ok_frames / track_evaluated_frame_count
            if track_evaluated_frame_count else None
        ),
        'ambiguity_margin': U11_AMBIGUITY_MARGIN,
    }
    save_json_atomically(u11_summary, U11_SUMMARY_PATH)
    display(u11_selections[[
        'video_id', 'group', 'status', 'reason', 'frame_index', 'candidate_id',
        'candidate_count', 'distance_margin', 'matches_manual',
        'track_ok_frame_count',
    ]])
    display(pd.DataFrame([u11_summary]))

## 5. 첫 후보 frame 시각 검수

초록색은 자동 선택, 주황색은 선택하지 않은 후보, 빨간색은 확인이 필요한 후보입니다. 표의 `matches_manual`과 그림을 함께 확인합니다.

In [ ]:
if u11_selections.empty and U11_SELECTION_PATH.exists():
    u11_selections = pd.read_csv(U11_SELECTION_PATH)
if u11_selections.empty:
    print('시각 검수 대기: 먼저 U11 자동 선택을 실행하세요.')
else:
    metadata_by_video = pilot_rows.set_index('video_id')
    figure, axes = plt.subplots(1, len(u11_selections), figsize=(18, 7), squeeze=False)
    for axis, selection in zip(axes.flat, u11_selections.itertuples(index=False), strict=True):
        if pd.isna(selection.frame_index):
            axis.set_title(f'{selection.video_id} | {selection.status}')
            axis.axis('off')
            continue
        metadata = metadata_by_video.loc[selection.video_id]
        candidates = pd.read_csv(
            U10_EVALUATION_ROOT / selection.video_id / 'plate_candidates.csv'
        )
        first_candidates = candidates.loc[
            candidates['frame_index'].eq(int(selection.frame_index))
        ]
        frame = read_video_frame(
            PROJECT_ROOT / str(metadata['path']), int(selection.frame_index)
        ).copy()
        for candidate in first_candidates.itertuples(index=False):
            if selection.status == 'confirmation_required':
                color = (0, 0, 255)
            elif int(candidate.candidate_id) == int(selection.candidate_id):
                color = (0, 255, 0)
            else:
                color = (0, 165, 255)
            start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
            end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
            cv2.rectangle(frame, start, end, color, 4)
            cv2.putText(
                frame, f'id={int(candidate.candidate_id)}', start,
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2,
            )
        axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axis.set_title(
            f'{selection.video_id} | {selection.status} | manual_match={selection.matches_manual}'
        )
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(U11_CONTACT_SHEET_PATH, dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    print(f'검수 그림: {U11_CONTACT_SHEET_PATH}')

## 6. 다음 checkpoint

1. `RUN_U11_AUTO_SELECTION=True`로 Task 1을 실행합니다.
2. 세 영상의 `matches_manual`, wrong auto selection, manual review rate와 track coverage를 확인합니다.
3. 자동 선택이 3/3이고 잘못된 선택이 0이면, Task 2에서 A/B/C 각 3영상의 첫 후보 frame만 추가 검수합니다.
4. 추가 검수 전에는 학습 모델·복합 점수·전체 54영상 라벨링을 만들지 않습니다.